# Mnemósine: asistente de mitología griega

Proyecto final de IA Generativa. Es un agente RAG con Google Gemini, ChromaDB y LangGraph que mantiene memoria de la conversación.

El nombre viene de Mnemósine, la titánide de la memoria y madre de las Musas, porque el agente tiene que recordar tanto sus fuentes como lo que se ha hablado.

| Componente | Tecnología | Archivo |
|---|---|---|
| LLM | Gemini (`gemini-3.5-flash-lite`) | `src/agent.py` |
| Embeddings | Gemini (`gemini-embedding-2`) | `src/knowledge_base.py` |
| Base vectorial | ChromaDB (persistente, similitud coseno) | `src/knowledge_base.py` |
| Agente y memoria | LangGraph + `MemorySaver` | `src/agent.py` |
| System prompt | | `src/prompts.py` (explicado en el README) |
| Parámetros | | `src/config.py` |

El código del agente está en `src/` para que este notebook y la app de Streamlit usen lo mismo. Aquí solo se carga, se prueba y se enseñan ejemplos.

## 0. Configuración del entorno

In [1]:
# Instalar dependencias (solo la primera vez):
# !pip install -r ../requirements-notebook.txt

In [2]:
import os
import sys
import textwrap
import uuid
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import DEFAULT_SETTINGS  # carga el .env de la raíz del proyecto

# La key se lee del .env
assert os.getenv("GOOGLE_API_KEY"), "Falta GOOGLE_API_KEY: copia .env.example a .env y añade tu key."
print("API key cargada desde .env")
print(f"   LLM:        {DEFAULT_SETTINGS.chat_model}")
print(f"   Embeddings: {DEFAULT_SETTINGS.embedding_model}")

API key cargada desde .env
   LLM:        gemini-3.5-flash-lite
   Embeddings: models/gemini-embedding-2


## 1. Dominio: mitología griega

Motivos para elegir este tema:
- Los mitos tienen muchas versiones (Hesíodo, Homero, Apolodoro...) y los LLM suelen mezclarlas o inventarse genealogías. Si las respuestas se basan en fuentes concretas y las citan, ese problema se reduce bastante.
- Se prestan a preguntas encadenadas («¿quién mató a Medusa?» y luego «¿y qué nació de su sangre?»), que sirven para probar la memoria.
- Las fuentes originales (Homero y Hesíodo) tienen traducciones de dominio público, así que se puede trabajar con fuentes primarias y no solo con resúmenes.

## 2. Corpus y preprocesado

La base de conocimiento tiene un documento por cada tipo de fuente:

| Documento | Tipo | Contenido | Idioma |
|---|---|---|---|
| `01_fuentes_primarias.txt` | Fuente primaria | Hesíodo, *Teogonía* (trad. Evelyn-White, 1914), y Homero, *Ilíada* y *Odisea* (trad. Luis Segalá, 1908-1910) | EN / ES |
| `02_manual_clasico.txt` | Manual clásico | Thomas Bulfinch, *The Age of Fable* (1855) | EN |
| `03_enciclopedia.txt` | Enciclopedia | 15 artículos de Wikipedia en español | ES |

- La fuente primaria recoge lo que contaban los propios griegos, con sus contradicciones; el manual ordena los mitos y la enciclopedia añade contexto moderno. Con las tres el agente puede decir «según Hesíodo...» y comparar versiones.
- Hay textos en inglés porque no hay traducción española de dominio público de la *Teogonía* ni un manual de mitología en español libre de derechos. Los embeddings de Gemini son multilingües (comparan significado, no palabras); en la sección 3 se comprueba.
- Licencias: Homero, Hesíodo y Bulfinch son de dominio público (Project Gutenberg); Wikipedia es CC BY-SA 4.0.

Preprocesado (`scripts/build_corpus.py`, se regenera con `python scripts/build_corpus.py`):
1. Gutenberg: quitar cabecera y licencia, rehacer los párrafos (el texto viene cortado a 70 columnas), eliminar números de verso, notas e ilustraciones y modernizar la ortografía de 1908 («á» pasa a «a»).
2. Nombres latinos a griegos: Segalá y Bulfinch usan los nombres romanos (Júpiter, Minerva, Ulises). Se añade el griego entre corchetes, «Minerva [Atenea]», para que una búsqueda por «Atenea» encuentre también esos pasajes.
3. Wikipedia: texto plano, sin secciones como Referencias o Enlaces externos.
4. Cada pasaje o artículo tiene un límite de palabras para que el total quede en unas 20 páginas.

In [3]:
WORDS_PER_PAGE = 500
total = 0
for path in sorted((PROJECT_ROOT / "data").glob("*.txt")):
    text = path.read_text(encoding="utf-8")
    words, sections = len(text.split()), text.count("Título: ")
    total += words
    print(f"{path.name:<26} {sections:>2} secciones  {words:>6} palabras  ~{words / WORDS_PER_PAGE:4.1f} págs.")
print(f"{'TOTAL':<26} {'':>12} {total:>6} palabras  ~{total / WORDS_PER_PAGE:4.1f} págs.")

01_fuentes_primarias.txt    8 secciones    3610 palabras  ~ 7.2 págs.
02_manual_clasico.txt       7 secciones    3552 palabras  ~ 7.1 págs.
03_enciclopedia.txt        15 secciones    3651 palabras  ~ 7.3 págs.
TOTAL                                    10813 palabras  ~21.6 págs.


In [4]:
# Ejemplo de la anotación de nombres en un pasaje de Homero (traducción de 1908)
text = (PROJECT_ROOT / "data" / "01_fuentes_primarias.txt").read_text(encoding="utf-8")
start = text.index("Título: Homero, Ilíada XXII")
print(text[start:start + 900])

Título: Homero, Ilíada XXII — La muerte de Héctor
Fuente: https://www.gutenberg.org/ebooks/57654
Tipo: Fuente primaria
Idioma: es
Obra: Homero, Ilíada (trad. Luis Segalá y Estalella, 1908)

Esto dicho, desenvainó la aguda espada, grande y fuerte, que llevaba en el costado. Y encogiéndose, se arrojó como el águila de alto vuelo se lanza a la llanura, atravesando las pardas nubes, para arrebatar la tierna corderilla o la tímida liebre; de igual manera arremetió Héctor, blandiendo la aguda espada. Aquiles embistióle, a su vez, con el corazón rebosante de feroz cólera: defendía su pecho con el magnífico escudo labrado, y movía el luciente casco de cuatro abolladuras, haciendo ondear las bellas y abundantes crines de oro que Vulcano [Hefesto] colocara en la cimera. Como el Véspero, que es el lucero más hermoso de cuantos


## 3. Base vectorial (ChromaDB + embeddings de Gemini)

`KnowledgeBase` carga los documentos, los divide en chunks, calcula los embeddings y los guarda en ChromaDB. Los parámetros están en `src/config.py`:

- Chunks de 1000 caracteres con 150 de solapamiento, que son uno o dos párrafos, lo justo para un episodio completo. Se corta primero por secciones (`##`) y si no, por párrafos.
- Cada chunk lleva delante el título de su artículo (`[Medusa] ...`). Un fragmento como «fue decapitada por Perseo» no dice de quién habla, así que sin el título el embedding pierde esa información.
- Similitud coseno, la métrica habitual para embeddings de texto.
- La colección se guarda en disco: la primera vez tarda unos 2 minutos en indexar (con la cuota gratuita de Gemini) y después se carga directamente.

In [5]:
from src.domain import ACTIVE_DOMAIN
from src.knowledge_base import KnowledgeBase

knowledge_base = KnowledgeBase(ACTIVE_DOMAIN.collection_name)
knowledge_base.load_or_build()  # force_rebuild=True para reindexar

for source in knowledge_base.describe():
    print(f"[{source.source_type:<15}] [{source.language}] {source.title:<48} {source.chunk_count} chunks")

[KB] Colección 'greek_mythology' cargada (90 chunks)
[Fuente primaria] [en] Hesíodo, Teogonía — El origen del mundo          3 chunks
[Fuente primaria] [en] Hesíodo, Teogonía — Crono devora a sus hijos     3 chunks
[Fuente primaria] [en] Hesíodo, Teogonía — Prometeo roba el fuego       2 chunks
[Fuente primaria] [en] Hesíodo, Teogonía — La Titanomaquia              2 chunks
[Fuente primaria] [es] Homero, Ilíada I — La cólera de Aquiles          4 chunks
[Fuente primaria] [es] Homero, Ilíada XXII — La muerte de Héctor        5 chunks
[Fuente primaria] [es] Homero, Odisea IX — El cíclope Polifemo          7 chunks
[Fuente primaria] [es] Homero, Odisea XII — Las sirenas                 4 chunks
[Manual clásico ] [en] Bulfinch — Prometeo y Pandora                    3 chunks
[Manual clásico ] [en] Bulfinch — Perseo y Medusa                       4 chunks
[Manual clásico ] [en] Bulfinch — Heracles                              4 chunks
[Manual clásico ] [en] Bulfinch — Teseo y el Minotauro  

### Comprobación de la búsqueda

Antes de conectar el LLM conviene ver que la búsqueda devuelve las fuentes correctas, porque si falla aquí el prompt no lo va a arreglar. Se prueban tres cosas:
- Una pregunta en español que debe encontrar pasajes en inglés (Hesíodo, Bulfinch).
- Una pregunta por «Atenea» que debe encontrar el pasaje de Homero donde pone «Minerva».
- Fuentes en español: Homero (Segalá) y Wikipedia.

In [6]:
test_queries = {
    "¿Qué surgió primero en el origen del mundo?": "Hesíodo (EN)",
    "¿Por qué Orfeo perdió a Eurídice?": "Bulfinch (EN)",
    "¿Cómo engañó Atenea a Héctor antes de su muerte?": "Homero, Ilíada XXII (dice «Minerva»)",
    "¿Cómo escapó Odiseo de la cueva del cíclope?": "Homero, Odisea IX (ES)",
    "¿Quién era la madre de las Musas?": "Wikipedia, Mnemósine (ES)",
}
for query, expected in test_queries.items():
    results = knowledge_base.search(query, k=3)
    found = "\n             ".join(f"{r.score:.2f}  [{r.document.metadata['language']}] {r.title}" for r in results)
    print(f"? {query}\n   esperado: {expected}\n   obtenido: {found}\n")

? ¿Qué surgió primero en el origen del mundo?
   esperado: Hesíodo (EN)
   obtenido: 0.75  [en] Hesíodo, Teogonía — El origen del mundo
             0.67  [es] Wikipedia — Caos (mitología)
             0.67  [en] Hesíodo, Teogonía — El origen del mundo



? ¿Por qué Orfeo perdió a Eurídice?
   esperado: Bulfinch (EN)
   obtenido: 0.77  [en] Bulfinch — Orfeo y Eurídice
             0.76  [en] Bulfinch — Orfeo y Eurídice
             0.75  [en] Bulfinch — Orfeo y Eurídice



? ¿Cómo engañó Atenea a Héctor antes de su muerte?
   esperado: Homero, Ilíada XXII (dice «Minerva»)
   obtenido: 0.68  [es] Homero, Ilíada XXII — La muerte de Héctor
             0.68  [es] Homero, Ilíada XXII — La muerte de Héctor
             0.67  [en] Bulfinch — La caída de Troya



? ¿Cómo escapó Odiseo de la cueva del cíclope?
   esperado: Homero, Odisea IX (ES)
   obtenido: 0.74  [es] Homero, Odisea IX — El cíclope Polifemo
             0.71  [es] Homero, Odisea IX — El cíclope Polifemo
             0.71  [es] Homero, Odisea IX — El cíclope Polifemo



? ¿Quién era la madre de las Musas?
   esperado: Wikipedia, Mnemósine (ES)
   obtenido: 0.77  [es] Wikipedia — Mnemósine
             0.66  [es] Wikipedia — Zeus
             0.65  [en] Hesíodo, Teogonía — El origen del mundo



## 4. System prompt

El prompt está en `src/prompts.py` y en el README se explica cada parte. Resumen:

| Parte | Qué dice | Motivo |
|---|---|---|
| Rol | Mnemósine, titánide de la memoria | Encaja con el tema y deja claro el ámbito |
| Fuentes | Qué es cada tipo de fuente, que hay pasajes en inglés que debe traducir y qué significa «Minerva [Atenea]» | Para que sepa cuánto peso dar a cada fuente |
| Citas | Usar primero la fuente primaria y citar autor y obra: `[Hesíodo, Teogonía]` | Menos alucinaciones y se puede comprobar cada dato |
| Si falta información | Puede usar conocimiento general marcado como «(fuera de mis fuentes)» | No se queda sin responder, pero se distingue lo que no viene de las fuentes |
| Versiones del mito | Dar las variantes por autor en vez de elegir una | Es un fallo frecuente de los LLM con mitología |
| Memoria | Resolver «él», «su padre», «y después» con los turnos anteriores | Para seguir la conversación |
| Fuera de dominio | No responder y sugerir una pregunta de mitología | Mantener el agente en su tema |
| Tono | Algo evocador pero preciso, de 1 a 4 párrafos | Que no suene a enciclopedia sin perder precisión |

In [7]:
from src.prompts import SYSTEM_PROMPT
print(SYSTEM_PROMPT)

# ROL
Eres Mnemósine, titánide de la memoria y madre de las nueve Musas. Actúas como experta en mitología griega: dioses, titanes, héroes, criaturas y el ciclo troyano.

# TUS FUENTES
Cada fragmento indica su título y su tipo:
- Fuente primaria: los textos antiguos (Homero, Hesíodo). Son la voz de los propios griegos y tienen la máxima autoridad.
- Manual clásico: Bulfinch (1855), que narra los mitos de forma ordenada.
- Enciclopedia: Wikipedia, que aporta síntesis y contexto moderno.
Algunos fragmentos están en inglés: tradúcelos al español al usarlos. Los textos antiguos usan a veces nombres latinos; el equivalente griego aparece entre corchetes, como en «Minerva [Atenea]». En tus respuestas escribe directamente el nombre griego («Atenea», «Eos»), sin copiar la anotación.

# CÓMO RESPONDES
1. Basa tus respuestas en los fragmentos recibidos. Si hay una fuente primaria sobre el tema, apóyate en ella antes que en las demás.
2. Cita entre corchetes la fuente de cada dato importante, con 

## 5. Agente LangGraph con RAG y memoria

```
START -> rewrite_query -> retrieve -> generate -> END
```

| Nodo | Qué hace | Motivo |
|---|---|---|
| `rewrite_query` | Reescribe la pregunta para que se entienda sin el historial | Si no, «¿y qué nació de su sangre?» se buscaría en ChromaDB sin saber que «su» es Medusa |
| `retrieve` | Recupera los 4 chunks más parecidos de ChromaDB | Todas las preguntas pasan por la base de conocimiento |
| `generate` | Gemini responde con el system prompt, los fragmentos y los últimos 10 mensajes | La respuesta se basa en las fuentes y en la conversación |

Memoria: el grafo se compila con `MemorySaver` como checkpointer. LangGraph guarda el estado de cada conversación con su `thread_id`, así que dos conversaciones no se mezclan. Al LLM solo se le pasan los últimos 10 mensajes para que una conversación larga no llene el contexto.

In [8]:
from src.agent import RagAgent

agent = RagAgent(knowledge_base, ACTIVE_DOMAIN.system_prompt)
print(agent.graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	rewrite_query(rewrite_query)
	retrieve(retrieve)
	generate(generate)
	__end__([<p>__end__</p>]):::last
	__start__ --> rewrite_query;
	retrieve --> generate;
	rewrite_query --> retrieve;
	generate --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 6. Ejemplos

`ask` imprime en cada turno la consulta usada para buscar (en las preguntas de seguimiento sale reformulada), la respuesta y las fuentes recuperadas con su puntuación.

In [9]:
def ask(question: str, thread_id: str) -> None:
    reply = agent.ask(question, thread_id)
    print(f"PREGUNTA: {question}")
    if reply.search_query != question:
        print(f"Consulta reformulada: {reply.search_query}")
    print("\nMNEMÓSINE:")
    for paragraph in reply.answer.split("\n"):
        print(textwrap.fill(paragraph, width=110) if paragraph else "")
    sources = ", ".join(f"{s['title']} ({s['score']:.2f})" for s in reply.sources)
    print(f"\nFuentes: {sources}")


conversation = str(uuid.uuid4())  # una conversación = un thread_id

### Ejemplo 1: pregunta sobre un héroe

In [10]:
ask("¿Cómo derrotó Perseo a Medusa?", conversation)

PREGUNTA: ¿Cómo derrotó Perseo a Medusa?

MNEMÓSINE:
Perseo logró derrotar a la monstruosa Medusa gracias a la ayuda de los dioses Hermes y Atenea, quienes lo
favorecieron prestándole sus sandalias aladas y un escudo brillante [Bulfinch — Perseo y Medusa]. Acercándose
mientras ella dormía y evitando mirarla de manera directa para no quedar petrificado, el héroe se guio
únicamente por la imagen reflejada en el escudo para cortarle la cabeza [Bulfinch — Perseo y Medusa]. Tras
esta hazaña, entregó el rostro de la gorgona a Atenea, quien lo fijó en el centro de su égida [Wikipedia —
Medusa (mitología)].

Fuentes: Bulfinch — Perseo y Medusa (0.76), Bulfinch — Perseo y Medusa (0.72), Wikipedia — Medusa (mitología) (0.71), Bulfinch — Perseo y Medusa (0.66)


### Ejemplo 2: pregunta que depende de la anterior

«Su sangre» solo se entiende si el agente recuerda que se hablaba de Medusa (ver la consulta reformulada).

In [11]:
ask("¿Y qué nació de su sangre?", conversation)

PREGUNTA: ¿Y qué nació de su sangre?
Consulta reformulada: ¿Qué nació de la sangre de Medusa?

MNEMÓSINE:
Cuando Perseo cercenó la cabeza de la gorgona, de su interior brotaron el gigante Crisaor y el célebre caballo
alado Pegaso [Wikipedia — Pegaso].

Fuentes: Bulfinch — Perseo y Medusa (0.70), Wikipedia — Pegaso (0.70), Bulfinch — Perseo y Medusa (0.70), Wikipedia — Medusa (mitología) (0.69)


### Ejemplo 3: titanes y olímpicos

In [12]:
ask("¿Cómo llegaron Zeus y sus hermanos a gobernar el mundo?", conversation)

PREGUNTA: ¿Cómo llegaron Zeus y sus hermanos a gobernar el mundo?

MNEMÓSINE:
Los hermanos de Zeus —Hades, Poseidón, Hestia, Deméter y Hera— llegaron al poder tras una gran contienda
contra los titanes, conocida como la Titanomaquia [Wikipedia — Titanomaquia, Wikipedia — Zeus]. Previamente,
el titán Crono había devorado a sus hijos al nacer para evitar ser derrocado, pero Rea, angustiada, logró
salvar a Zeus [Hesíodo, Teogonía — Crono devora a sus hijos].

Cuando Zeus alcanzó la madurez, obligó a su padre a vomitar a sus hermanos y liberó del Tártaro a los Cíclopes
y a los Hecatónquiros para aliarse con ellos [Wikipedia — Titanomaquia]. Tras derrotar a los titanes en esa
terrible guerra, los tres hermanos varones se repartieron el dominio del mundo: Zeus obtuvo el cielo y el
poder supremo, Poseidón el mar y Hades el mundo subterráneo [Wikipedia — Zeus, Wikipedia — Poseidón].

Fuentes: Wikipedia — Zeus (0.71), Wikipedia — Poseidón (0.69), Wikipedia — Titanomaquia (0.69), Hesíodo, Teogon

### Ejemplo 4: referencia a una respuesta anterior

«Aquel castigo a los titanes» se refiere a la respuesta anterior, y «el titán que ayudó a los humanos» hay que relacionarlo con Prometeo.

In [13]:
ask("Además de aquel castigo a los titanes, ¿a qué titán castigó Zeus por ayudar a los humanos, y cómo?", conversation)

PREGUNTA: Además de aquel castigo a los titanes, ¿a qué titán castigó Zeus por ayudar a los humanos, y cómo?
Consulta reformulada: ¿Qué castigo impuso Zeus al titán que ayudó a los humanos y en qué consistió?

MNEMÓSINE:
El titán castigado por Zeus fue Prometeo, hijo de Jápeto, debido a que desafió la voluntad del soberano del
Olimpo al robar el fuego en el interior de una cañaheja para entregárselo a los mortales [Hesíodo, Teogonía —
Prometeo roba el fuego, Bulfinch — Prometeo y Pandora]. Para vengarse de esta osadía y del beneficio que
supuso para la humanidad, Zeus ordenó al dios Hefesto —el dios cojo— moldear a partir de la tierra a la
primera mujer, Pandora, a quien dotaron de dones los dioses con el fin de enviar un mal a los hombres
[Hesíodo, Teogonía — Prometeo roba el fuego, Bulfinch — Prometeo y Pandora]. *(Fuera de mis fuentes: además de
este engaño con Pandora, la tradición mitológica añade que Zeus condenó a Prometeo a ser encadenado a una roca
en el Cáucaso, donde un águi

### Ejemplo 5: fuente primaria (Homero)

In [14]:
ask("¿Cómo escapó Odiseo del cíclope Polifemo?", conversation)

PREGUNTA: ¿Cómo escapó Odiseo del cíclope Polifemo?

MNEMÓSINE:
Odiseo logró escapar del cíclope Polifemo astutamente tras emborracharlo con un vino generoso [Homero, Odisea
IX — El cíclope Polifemo]. Mientras el monstruo dormía pesadamente, el héroe y sus hombres calentaron una
estaca de olivo al fuego y se la clavaron en el único ojo para cegarlo [Homero, Odisea IX — El cíclope
Polifemo].

Cuando Polifemo, gimiendo de dolor, retiró el gran peñasco de la entrada para atrapar a quien saliera, Odiseo
ideó un nuevo artificio [Homero, Odisea IX — El cíclope Polifemo]. Ató a los carneros del rebaño de tres en
tres con mimbres, colocando a un compañero bajo el animal del centro, mientras él mismo se aferró por la
espalda a la abundante lana del carnero más grande y robusto [Homero, Odisea IX — El cíclope Polifemo]. De
este modo, al salir las reses a pastar con las primeras luces de la Aurora, el héroe y sus hombres
consiguieron burlar las manos del monstruo y escapar sanos y salvos [Homero,

### Ejemplo 6: «después» y «él»

Solo se entiende si se sabe que se hablaba de Odiseo y su viaje.

In [15]:
ask("¿Y qué peligro le advirtió Circe que encontraría después, y cómo lo superó él?", conversation)

PREGUNTA: ¿Y qué peligro le advirtió Circe que encontraría después, y cómo lo superó él?
Consulta reformulada: ¿Qué peligro le advirtió Circe a Odiseo que encontraría después y cómo lo superó Odiseo?

MNEMÓSINE:
Circe y la propia voz de la divina hechicera advirtieron a Odiseo que se encontraría primero con las Sirenas,
criaturas que hechizan con su sonoro canto a cuantos hombres se acercan, rodeadas por un montón de huesos de
humanos cuya piel se va consumiendo [Homero, Odisea XII — Las sirenas]. Para superarlas sin perecer, Odiseo
tapó las orejas de sus compañeros con cera blanda y ordenó que lo atasen fuertemente de pies y manos al mástil
de la embarcación, prohibiendo que lo soltaran por mucho que él lo suplicara [Homero, Odisea XII — Las
sirenas].

Posteriormente, la ruta presentaba una terrible bifurcación sin un camino seguro: a un lado se alzaban las
rocas Erráticas, contra las que rugían las inmensas olas de Anfitrite y donde las tempestades y el fuego
destruían naves y hombre

### Ejemplo 7: pregunta fuera del dominio

Según el system prompt no debe contestarla y tiene que proponer una pregunta de mitología.

In [16]:
ask("¿Cuál es la capital de Australia?", conversation)

PREGUNTA: ¿Cuál es la capital de Australia?

MNEMÓSINE:
Mi memoria guarda únicamente los mitos, dioses, héroes y leyendas de la antigua Grecia, por lo que no poseo
respuestas sobre geografía moderna.

Si lo deseas, puedes preguntarme acerca de algún viaje mitológico, como las pruebas que afrontó Jasón con los
argonautas, o sobre los peligros a los que se enfrentó Odiseo tras dejar atrás a las Sirenas.

Fuentes: Wikipedia — Guerra de Troya (0.48), Wikipedia — Hera (0.48), Wikipedia — Minotauro (0.48), Wikipedia — Zeus (0.47)


### Ejemplo 8: la memoria va por conversación

La respuesta a esta pregunta solo está en la conversación, no en los documentos, así que la búsqueda en ChromaDB no sirve de nada aquí.

- 8a: en la misma conversación debería recordar los personajes de los últimos turnos (Zeus, Prometeo, Odiseo).
- 8b: con un `thread_id` nuevo no hay historial y debería decir que no lo sabe.

Nota: al LLM solo le llegan los últimos 10 mensajes (`max_history_messages`), por eso lo del ejemplo 1 (Perseo) ya no lo ve. Es a propósito, para no llenar el contexto en conversaciones largas. Se podría mejorar resumiendo los turnos antiguos (ver README).

In [17]:
ask("Para terminar: ¿por qué personajes te he preguntado en mis últimas preguntas?", conversation)

PREGUNTA: Para terminar: ¿por qué personajes te he preguntado en mis últimas preguntas?
Consulta reformulada: ¿Por qué personajes mitológicos griegos he preguntado en mis últimas preguntas de esta conversación?

MNEMÓSINE:
En tus mensajes anteriores me preguntaste por Prometeo, Odiseo, el cíclope Polifemo y la hechicera Circe.

Fuentes: Wikipedia — Pegaso (0.66), Wikipedia — Zeus (0.66), Wikipedia — Mnemósine (0.66), Hesíodo, Teogonía — El origen del mundo (0.65)


In [18]:
ask("Para terminar: ¿por qué personajes te he preguntado en mis últimas preguntas?", thread_id=str(uuid.uuid4()))

PREGUNTA: Para terminar: ¿por qué personajes te he preguntado en mis últimas preguntas?

MNEMÓSINE:
Ya no lo recuerdo.

Fuentes: Bulfinch — La caída de Troya (0.59), Wikipedia — Minotauro (0.58), Wikipedia — Mnemósine (0.58), Homero, Odisea IX — El cíclope Polifemo (0.58)


### Historial guardado en LangGraph

In [19]:
history = agent.history(conversation)
print(f"{len(history)} mensajes en la conversación '{conversation[:8]}'\n")
for number, message in enumerate(history, start=1):
    role = "Pregunta" if message["role"] == "user" else "Respuesta"
    print(f"{number:>2}. {role + ':':<10} {message['content'][:95].replace(chr(10), ' ')}...")

16 mensajes en la conversación 'de779710'

 1. Pregunta:  ¿Cómo derrotó Perseo a Medusa?...
 2. Respuesta: Perseo logró derrotar a la monstruosa Medusa gracias a la ayuda de los dioses Hermes y Atenea, ...
 3. Pregunta:  ¿Y qué nació de su sangre?...
 4. Respuesta: Cuando Perseo cercenó la cabeza de la gorgona, de su interior brotaron el gigante Crisaor y el ...
 5. Pregunta:  ¿Cómo llegaron Zeus y sus hermanos a gobernar el mundo?...
 6. Respuesta: Los hermanos de Zeus —Hades, Poseidón, Hestia, Deméter y Hera— llegaron al poder tras una gran ...
 7. Pregunta:  Además de aquel castigo a los titanes, ¿a qué titán castigó Zeus por ayudar a los humanos, y có...
 8. Respuesta: El titán castigado por Zeus fue Prometeo, hijo de Jápeto, debido a que desafió la voluntad del ...
 9. Pregunta:  ¿Cómo escapó Odiseo del cíclope Polifemo?...
10. Respuesta: Odiseo logró escapar del cíclope Polifemo astutamente tras emborracharlo con un vino generoso [...
11. Pregunta:  ¿Y qué peligro le advirtió Cir

## 7. Chat interactivo

Ejecuta la celda y escribe preguntas. `nueva` empieza otra conversación y `salir` termina.

También hay interfaz web: `streamlit run app/streamlit_app.py`

In [ ]:
thread_id = str(uuid.uuid4())
print("Mnemósine te escucha. Escribe 'nueva' para otra conversación o 'salir' para terminar.")
while True:
    question = input("\nTú: ").strip()
    if question.lower() == "salir":
        print("Que las Musas te acompañen.")
        break
    if question.lower() == "nueva":
        thread_id = str(uuid.uuid4())
        print("Nueva conversación.")
        continue
    if question:
        ask(question, thread_id)